# Text-to-CAD Fine-tuning with CodeT5-small on Google Colab (Free T4 GPU)

This notebook fine-tunes **Salesforce/codet5-small** (~60M params) to generate CAD specifications from natural language.

**Runtime:** ~30-60 minutes on Colab Free (T4 16GB)
**Cost:** $0

In [ ]:
# Check GPU availability
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Clone the project repository
!git clone https://github.com/nagulapallidheeraj08-star/ai-cad-project.git 2>/dev/null || echo "Repo already exists"
%cd ai-cad-project

In [ ]:
# Install dependencies
!pip install -q transformers accelerate torch cadquery datasets tqdm

# Verify imports
import transformers
import torch
import cadquery
print(f"Transformers: {transformers.__version__}")
print(f"Torch: {torch.__version__}")
print(f"CadQuery: {cadquery.__version__}")

In [ ]:
# Generate synthetic training data (10K samples)
!python generate_synthetic_data.py --num-samples 10000 --output data/synthetic_cad_samples.json

In [ ]:
# Start fine-tuning
!python train_codet5.py

In [ ]:
# Verify model was saved - search all possible locations
import os

# Check current directory
print(f"CWD: {os.getcwd()}")
print(f"CWD contents: {os.listdir('.')}")

# Search for model directories
possible_paths = [
    "./cad_codet5_finetuned",
    "./ai-cad-project/cad_codet5_finetuned",
    "/content/cad_codet5_finetuned",
    "/content/ai-cad-project/cad_codet5_finetuned",
]

found = False
for path in possible_paths:
    if os.path.exists(path):
        print(f"\nFound model at: {path}")
        print(f"Contents: {os.listdir(path)}")
        for f in os.listdir(path):
            fp = os.path.join(path, f)
            print(f"  {f}: {os.path.getsize(fp)} bytes")
        found = True
        MODEL_DIR = path
        break

if not found:
    print("\nModel not found in expected locations")
    print("Searching for model files...")
    for root, dirs, files in os.walk("."):
        for f in files:
            if f in ["pytorch_model.bin", "model.safetensors", "config.json"]:
                print(f"  Found: {os.path.join(root, f)}")

In [ ]:
# Test the fine-tuned model - only if valid model found
import json
import torch
import os
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Check if we have a valid MODEL_DIR with required files
required_files = ["config.json", "pytorch_model.bin"]
model_ready = False

if 'MODEL_DIR' in locals() and os.path.exists(MODEL_DIR):
    missing = [f for f in required_files if not os.path.exists(os.path.join(MODEL_DIR, f))]
    if not missing:
        model_ready = True
    else:
        print(f"Model dir exists but missing files: {missing}")

if not model_ready:
    print("Model not found or incomplete. Cannot test.")
    print("Check the 'Verify model' cell output above.")
    print("Training may have failed or not completed.")
else:
    print(f"Loading model from: {MODEL_DIR}")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_DIR, local_files_only=True)

    if torch.cuda.is_available():
        model = model.cuda()

    def generate_cad(text):
        inputs = tokenizer(text, return_tensors="pt", max_length=512, truncation=True)
        if torch.cuda.is_available():
            inputs = {k: v.cuda() for k, v in inputs.items()}
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=512, num_beams=4)
        return tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Test cases
    test_cases = [
        "Create a 50mm x 30mm x 20mm block with a 10mm hole",
        "Make a 40mm cube with 3mm fillets",
        "Design a cylinder 100mm long, 25mm diameter with 5mm keyway",
        "Create a hollow cylinder 30mm OD, 20mm ID, 50mm height",
        "Design a bracket 60x40x5mm with two 8mm holes 45mm apart and 30mm flange"
    ]

    for tc in test_cases:
        result = generate_cad(tc)
        print(f"Input:  {tc}")
        print(f"Output: {result}")
        print("-" * 80)

In [ ]:
# Optional: Upload model to Hugging Face Hub (free)
# Run this if you want to save/share your model

# from huggingface_hub import HfApi, login
# login()  # Will prompt for token
# model.push_to_hub("your-username/codet5-cad-finetuned")
# tokenizer.push_to_hub("your-username/codet5-cad-finetuned")
# print("Model uploaded to Hugging Face Hub!")

## Next Steps

1. **Download model**: In Colab file browser, right-click `cad_codet5_finetuned/` -> Download
2. **Use locally**: Copy to your project and update `text_to_cad.py` to load from `./cad_codet5_finetuned`
3. **Improve**: Generate more synthetic data (50K+), train longer, try CodeT5-base (220M)
4. **Export CAD**: Use the `CADGenerator` class to convert model output to STEP files